# HydroSwift Python Usage Examples
---

## Table of Contents

1. [Setup & Version Check](#1-setup--version-check)
2. [Python API Overview](#2-python-api-overview)
3. [WRIS Data: Variables, Basins & Stations](#3-wris-data-variables-basins--stations)
4. [WRIS Data: Using tables to download with `fetch()`](#4-wris-data-using-tables-to-download-with-fetch)
5. [WRIS Data: Download with `wris.download()`](#5-wris-data-download-with-wrisdownload)
6. [CWC Data: Basins & Stations](#6-cwc-data-basins--stations)
7. [CWC Data: Using tables to download with `fetch()`](#7-cwc-data-using-tables-to-download-with-fetch)
8. [CWC Data: Download with `cwc.download()`](#8-cwc-data-download-with-cwcdownload)
9. [Post-Processing: Merging into GeoPackage](#9-post-processing-merging-into-geopackage)
10. [Post-Processing: Generating Time-Series Plots](#10-post-processing-generating-time-series-plots)
11. [Full Example Workflow](#11-full-example-workflow)
12. [API Quick Reference Table](#12-api-quick-reference-table)
13. [Misc.](#13-misc)

## 1. Setup & Version Check

Install HydroSwift if needed, then import it.

In [ ]:
# Uncomment to install:
# !pip install hydroswift

import hydroswift
print(hydroswift.__version__)


View the Python API help text at any time:

In [ ]:
hydroswift.help()

Or the CLI help text from Python (useful as a reference while scripting):

In [ ]:
hydroswift.cli_help()

## 2. Python API Overview

HydroSwift has two download functions available:

| Workflow | When to use | Entry point |
|---|---|---|
| **Table-driven** | When you want to *discover* basins/stations first, inspect, subset, then download | `wris.basins()` / `wris.stations()` / `cwc.basins()` / `cwc.stations()` → `fetch()` |
| **Explicit download** | When you already know exact basins, stations, and variables | `wris.download()` / `cwc.download()` |

Both workflows support the same output options (`merge`, `plot`, `format`, `output_dir`, etc.).

> **Important!:** `fetch()` accepts only HydroSwift-generated tables — not arbitrary DataFrames.
> `wris.download()` and `cwc.download()` accept explicit strings/lists, not DataFrames.


## 3. WRIS Data: Variables, Basins & Stations

Before downloading, you can explore what data is available using the discovery methods.
These return `SwiftTable` objects (a pandas DataFrame subclass with `.attrs` metadata)
that can be passed directly into `fetch()`.


### 3a. `wris.variables()` : List all supported WRIS variables

In [ ]:
# Returns a table with flag, dataset_code, canonical_name, and aliases for each variable.
variables = hydroswift.wris.variables()
variables


### 3b. `wris.basins()`: List all WRIS basins (no variable filter)

In [ ]:
# Returns one row per basin — useful for exploration.
all_basins = hydroswift.wris.basins()
all_basins


### 3c. `wris.basins(variable=...)`: Basin table filtered by variable(s)

When you pass a `variable` argument, you get one row per `(basin, variable)` pair.
This table can then be directly passed to `fetch()`.


In [ ]:
# One row per (basin, variable) pair — ready for fetch().
basins_wris = hydroswift.wris.basins(variable=['discharge', 'rainfall', 'sediment'])
basins_wris


In [ ]:
# Inspect table metadata — fetch() uses .attrs to determine dispatch type.
print(basins_wris.attrs)


### 3d. `wris.stations()` :  Station-level discovery

Use this to discover individual gauge stations for specific basins and variables.
Returns station metadata including coordinates, river, and basin.

> ⚠️ `state` is **not supported** for WRIS station filtering; passing it will result in a `ValueError`.


In [ ]:
# Discover discharge and sediment stations in Godavari and Narmada.
stations_wris = hydroswift.wris.stations(
    basin=['Godavari', 'Narmada'],
    variable=['discharge', 'sediment'],
    delay=0.25,   # seconds between API requests (be polite to the server)
)
stations_wris

In [ ]:
# Typical columns: station_code, station_name, latitude, longitude, river, basin, variable
print(stations_wris.columns.tolist())
print(f'Total stations found: {len(stations_wris)}')


### 3e. Subsetting the discovery table before downloading

One of the key advantages of the table-driven workflow is that you can inspect,
filter, or sort the discovery table before committing to a download.


In [ ]:
# Example: filter basins table to only Cauvery and Godavari
basins_subset = basins_wris[basins_wris['basin'].isin(['Cauvery', 'Godavari'])]
basins_subset


In [ ]:
# Example: filter station table to the first 5 stations alphabetically
stations_subset = stations_wris.sort_values('station_name').head(5)
stations_subset


## 4. WRIS Data: Using tables to download with `fetch()`


`hydroswift.fetch()` is the bridge between discovery tables and actual downloads.
It dispatches automatically based on the table type encoded in `.attrs`:

- WRIS basin table (`basin` + `variable` columns) → basin/variable dispatch
- WRIS station table (`station_code` column) → station-level dispatch
- CWC basin table → per-basin station expansion
- CWC station table (`code` column) → station-level dispatch


### 4a. `fetch()` from a WRIS basin table

In [ ]:
# Step 1 — get the basin table for the variables we want
basins_wris = hydroswift.wris.basins(variable=['sediment', 'water_level'])

# Step 2 — subset to basins of interest
basins_wris_subset = basins_wris[basins_wris['basin'].isin(['Cauvery', 'Godavari'])]

# Step 3 — download via fetch()
gdf_wris_basin = hydroswift.fetch(
    basins_wris_subset,
    output_dir='data/fetch_wris_basins',
    start_date='2024-04-01',
    end_date='2024-06-30',
    merge=True,      # merge all downloaded files into a single GeoDataFrame
    plot=True,       # generate station-wise time-series plots
    quiet=False,     # show download progress
    overwrite=True,
)
gdf_wris_basin


### 4b. `fetch()` from a WRIS station table

In [ ]:
# Step 1 — discover stations
stations_wris = hydroswift.wris.stations(
    basin=['Godavari', 'Narmada'],
    variable=['solar', 'sediment'],
)
stations_wris


In [ ]:
# Step 2 — optionally subset
stations_wris_subset = stations_wris[stations_wris['basin'] == 'Godavari']

# Step 3 — download
# If data is unavailable for a station/period, it is skipped gracefully.
gdf_wris_stations = hydroswift.fetch(
    stations_wris_subset,
    output_dir='data/fetch_wris_stations',
    start_date='2024-04-01',
    end_date='2024-09-30',
    merge=True,
    plot=True,
    quiet=False,
    overwrite=True,
    delay=0.25,   # pace WRIS API requests
)
gdf_wris_stations


### 4c. Inspecting the returned GeoDataFrame

In [ ]:
# The returned GeoDataFrame contains merged station data with geometry.
print(gdf_wris_stations.shape)
print(gdf_wris_stations.crs)
gdf_wris_stations.head()


## 5. WRIS Data : Download with `wris.download()`

Use `wris.download()` when you already know the exact basins, stations, and variables.
This skips the discovery step and downloads directly.

> **Note:** Inputs must be strings/ints/lists — **not DataFrames**.


### 5a. Single basin, single variable

In [ ]:
gdf = hydroswift.wris.download(
    basin='Godavari',
    variable='discharge',
    start_date='2024-01-01',
    end_date='2024-06-30',
    output_dir='data/wris_explicit',
    format='csv',
    overwrite=True,
    merge=True,
    plot=False,
    quiet=False,
)
gdf


### 5b. Multiple basins, multiple variables

In [ ]:
gdf_multi = hydroswift.wris.download(
    basin=['Krishna', 'Tapi'],
    variable=['discharge', 'solar'],
    start_date='2024-01-01',
    end_date='2024-03-31',
    output_dir='data/wris_multi',
    format='csv',
    overwrite=True,
    merge=True,
    plot=True,
    quiet=False,
)
gdf_multi


### 5c. Download using the WRIS numeric basin ID

In [ ]:
# Basin IDs: 5 = Godavari, 6 = Krishna (use wris.basins() to see the full list)
gdf_by_id = hydroswift.wris.download(
    basin=6,
    variable='rainfall',
    start_date='2023-06-01',
    end_date='2023-09-30',
    output_dir='data/wris_by_id',
    overwrite=True,
    merge=True,
)
gdf_by_id


### 5d. Download with station-level filtering

In [ ]:
# You can narrow the download to specific station codes within a basin.
gdf_station_filter = hydroswift.wris.download(
    basin='Godavari',
    variable='discharge',
    station=['SOME_STATION_CODE'],  # replace with actual code from wris.stations()
    start_date='2024-01-01',
    end_date='2024-06-30',
    output_dir='data/wris_station_filter',
    overwrite=True,
    merge=True,
)
gdf_station_filter


### 5e. Save as Excel instead of CSV

In [ ]:
hydroswift.wris.download(
    basin='Narmada',
    variable=['temperature', 'humidity'],
    start_date='2024-01-01',
    end_date='2024-03-31',
    output_dir='data/wris_xlsx',
    format='xlsx',          # 'csv' (default) or 'xlsx'
    overwrite=True,
    merge=False,            # skip merge; do it manually later with merge_only()
    quiet=True,             # suppress progress output
)


## 6. CWC Data: Basins & Stations

CWC provides **near real-time water level** data at gauge stations across India.
It is water-level only — no multi-variable selection is needed.


### 6a. `cwc.basins()`: List all CWC basins with station counts

In [ ]:
# Returns basin name and station_count. Fetch-ready for basin-level downloads.
basins_cwc = hydroswift.cwc.basins()
basins_cwc


In [ ]:
# Refresh station metadata from the live CWC API before returning the table.
# Use this if you suspect the packaged metadata is outdated.
basins_cwc_fresh = hydroswift.cwc.basins(refresh=True)
basins_cwc_fresh


### 6b. `cwc.stations()`: Station-level metadata

Filter by basin, river, state, or station code. Filters can be combined.
Returns station metadata including `code`, `name`, `basin`, `river`, `state`.


In [ ]:
# All CWC stations (no filter)
all_cwc_stations = hydroswift.cwc.stations()
print(f'Total CWC stations: {len(all_cwc_stations)}')
all_cwc_stations.head()


In [ ]:
# Filter by basin
stations_cwc_basin = hydroswift.cwc.stations(basin=['Narmada', 'Tapi'])
stations_cwc_basin


In [ ]:
# Filter by basin AND state (filters are intersected)
stations_cwc_state = hydroswift.cwc.stations(
    basin=['Narmada', 'Tapi'],
    state=['Gujarat', 'Maharashtra'],
)
stations_cwc_state


In [ ]:
# Filter by river name
stations_cwc_river = hydroswift.cwc.stations(river='Narmada')
stations_cwc_river


In [ ]:
# Filter by specific station codes
stations_cwc_codes = hydroswift.cwc.stations(station=['040-CDJAPR', '032-LGDHYD'])
stations_cwc_codes


In [ ]:
# Subset by station name before downloading
stations_cwc_subset = stations_cwc_state[stations_cwc_state['name'].isin(['BODELI', 'Bharuch'])]
stations_cwc_subset


## 7. CWC Data : Using tables to download with `fetch()`

`fetch()` accepts both CWC basin tables and CWC station tables:
- **Basin table** (`hydroswift.cwc.basins()`) → expands to all stations per basin, then downloads
- **Station table** (`hydroswift.cwc.stations()`) → downloads the specific stations in the table


### 7a. `fetch()` from a CWC basin table

In [ ]:
# Step 1 — get basin table and subset
basins_cwc = hydroswift.cwc.basins()
basins_cwc_subset = basins_cwc[basins_cwc['basin'].isin(['Narmada', 'Tapi'])]
basins_cwc_subset


In [ ]:
# Step 2 — download
gdf_cwc_basin = hydroswift.fetch(
    basins_cwc_subset,
    output_dir='data/fetch_cwc_basins',
    start_date='2024-04-01',
    end_date='2024-06-30',
    merge=True,
    plot=True,
    quiet=False,
    overwrite=True,
)
gdf_cwc_basin


### 7b. `fetch()` from a CWC station table

In [ ]:
# Step 1 — discover and subset stations
stations_cwc = hydroswift.cwc.stations(
    basin=['Narmada', 'Tapi'],
    state=['Gujarat', 'Maharashtra'],
)
stations_cwc_subset = stations_cwc[stations_cwc['name'].isin(['BODELI', 'Bharuch'])]

# Step 2 — download
gdf_cwc_stations = hydroswift.fetch(
    stations_cwc_subset,
    output_dir='data/fetch_cwc_stations',
    start_date='2024-04-01',
    end_date='2024-06-30',
    merge=True,
    plot=True,
    quiet=False,
    overwrite=True,
)
gdf_cwc_stations


### 7c. `fetch()` with metadata refresh

In [ ]:
# Pass refresh=True to re-fetch the CWC station registry before downloading.
gdf_cwc_refreshed = hydroswift.fetch(
    basins_cwc_subset,
    output_dir='data/fetch_cwc_refreshed',
    start_date='2024-04-01',
    end_date='2024-06-30',
    merge=True,
    refresh=True,    # refresh CWC station metadata from live API before dispatch
    overwrite=True,
)


## 8. CWC Data : Download with `cwc.download()`

Use `cwc.download()` when you already know the exact basins or station codes.
If both `basin` and `station` are provided, HydroSwift downloads the **intersection**.


### 8a. Download all CWC stations in specified basins

In [ ]:
gdf_cwc = hydroswift.cwc.download(
    basin=['Narmada', 'Tapi'],
    start_date='2024-01-01',
    end_date='2024-03-31',
    output_dir='data/cwc_explicit',
    format='csv',
    overwrite=True,
    merge=True,
    plot=True,
    quiet=False,
)
gdf_cwc.head()


### 8b. Download specific CWC station codes

In [ ]:
gdf_cwc_codes = hydroswift.cwc.download(
    station=['040-CDJAPR', '032-LGDHYD'],
    start_date='2024-01-01',
    end_date='2024-06-30',
    output_dir='data/cwc_stations',
    format='csv',
    overwrite=True,
    merge=True,
)
gdf_cwc_codes


### 8c. Intersection : basin filter + specific stations

In [ ]:
# Downloads only stations that match BOTH the basin AND the station list.
gdf_cwc_intersect = hydroswift.cwc.download(
    basin=['Krishna'],
    station=['032-LGDHYD'],
    start_date='2024-01-01',
    end_date='2024-03-31',
    output_dir='data/cwc_intersection',
    merge=True,
    overwrite=True,
)
gdf_cwc_intersect


### 8d. Download CWC data as Excel with metadata refresh

In [ ]:
hydroswift.cwc.download(
    basin=['Godavari'],
    start_date='2024-06-01',
    end_date='2024-09-30',
    output_dir='data/cwc_xlsx',
    format='xlsx',
    overwrite=True,
    merge=False,
    quiet=True,
    refresh=True,
)


## 9. Post-Processing: Merging into GeoPackage

`hydroswift.merge_only()` reads previously downloaded CSV/XLSX files from
`input_dir` and merges them into a **GeoPackage (`.gpkg`)** file.
Use this when you downloaded with `merge=False` and want to merge later.

| Parameter | Description |
|---|---|
| `input_dir` | Folder containing existing HydroSwift output files |
| `output_dir` | Where the `.gpkg` will be written (optional) |
| `mode` | `'wris'` or `'cwc'` |
| `variable` | WRIS only — subset of variables to merge (omit to merge all) |


### 9a. Merge CWC output

In [ ]:
gdf_merged_cwc = hydroswift.merge_only(
    mode='cwc',
    input_dir='data/cwc_explicit',
    output_dir='data/merged_cwc',
)
# Returns a GeoDataFrame and writes a .gpkg to output_dir.
gdf_merged_cwc


### 9b. Merge WRIS output from all variables

In [ ]:
gdf_merged_wris = hydroswift.merge_only(
    mode='wris',
    input_dir='data/fetch_wris_basins',
    output_dir='data/merged_wris',
    # variable not specified → merges ALL variables found in input_dir
)
gdf_merged_wris


### 9c. Merge WRIS output for a subset of variables

In [ ]:
gdf_merged_subset = hydroswift.merge_only(
    mode='wris',
    input_dir='data/fetch_wris_basins',
    output_dir='data/merged_wris_subset',
    variable=['sediment', 'water_level'],  # only merge these two variables
)
gdf_merged_subset


### 9d. Merge without writing to disk (in-memory)

In [ ]:
# Omit output_dir to merge in memory only (no .gpkg written).
gdf_in_memory = hydroswift.merge_only(
    mode='cwc',
    input_dir='data/cwc_explicit',
)
gdf_in_memory


## 10. Post-Processing: Generating Time-Series Plots

`hydroswift.plot_only()` reads existing HydroSwift output files and generates
station-wise time-series plots as PNG (default) or SVG.

| Parameter | Description |
|---|---|
| `input_dir` | Folder with HydroSwift output files |
| `output_dir` | Where plot images will be saved |
| `mode` | `'wris'` or `'cwc'` |
| `variable` | WRIS only — subset of variables to plot |
| `plot_svg` | `True` to save SVG in addition to PNG |
| `moving_average` | `True` to add a moving-average overlay (default window: 30) |
| `window` | Explicit window size for the moving average |


### 10a. Plot CWC data (PNG)

In [ ]:
hydroswift.plot_only(
    mode='cwc',
    input_dir='data/cwc_explicit',
    output_dir='plots/cwc',
    plot_svg=False,
)


### 10b. Plot CWC data with moving average overlay

In [ ]:
hydroswift.plot_only(
    mode='cwc',
    input_dir='data/cwc_explicit',
    output_dir='plots/cwc_ma',
    moving_average=True,
    window=5,    # 5-sample moving average window
    plot_svg=False,
)


### 10c. Plot WRIS data for all variables, with SVG output

In [ ]:
hydroswift.plot_only(
    mode='wris',
    input_dir='data/fetch_wris_basins',
    output_dir='plots/wris',
    plot_svg=True,    # saves both PNG and SVG
)


### 10d. Plot WRIS data for a subset of variables, 30-day moving average

In [ ]:
hydroswift.plot_only(
    mode='wris',
    input_dir='data/fetch_wris_basins',
    output_dir='plots/wris_ma',
    variable=['sediment', 'water_level'],
    moving_average=True,
    window=30,
    plot_svg=True,
)


## 11. Full Example Workflow

These examples show complete workflows from discovery → download → merge → plot.


### 11a. Using tables ; WRIS

Recommended pattern for reproducible notebook workflows:
1. Discover basins/stations
2. Inspect and subset
3. Download with `fetch()`
4. Post-process with `merge_only()` / `plot_only()`


In [ ]:
import hydroswift

# 1. Discover
basins = hydroswift.wris.basins(variable=['discharge', 'rainfall', 'temperature'])

# 2. Subset to Krishna basin
subset = basins[basins['basin'] == 'Krishna']

# 3. Download (merge=False so we can post-process separately)
hydroswift.fetch(
    subset,
    output_dir='data/pipeline_wris',
    start_date='2022-01-01',
    end_date='2024-12-31',
    format='csv',
    overwrite=True,
    merge=False,
    plot=False,
    quiet=False,
)


In [ ]:
# 4a. Merge into GeoPackage
gdf = hydroswift.merge_only(
    mode='wris',
    input_dir='data/pipeline_wris',
    output_dir='data/pipeline_wris_merged',
)
print(f'Merged GeoDataFrame: {gdf.shape}')
gdf.head()


In [ ]:
# 4b. Generate plots with 7-day moving average
hydroswift.plot_only(
    mode='wris',
    input_dir='data/pipeline_wris',
    output_dir='data/pipeline_wris_plots',
    moving_average=True,
    window=7,
    plot_svg=True,
)


### 11b. Using tables : CWC


In [ ]:
# 1. Discover CWC basins
cwc_basins = hydroswift.cwc.basins()

# 2. Subset to Godavari
cwc_subset = cwc_basins[cwc_basins['basin'] == 'Godavari']

# 3. Download
hydroswift.fetch(
    cwc_subset,
    output_dir='data/pipeline_cwc',
    start_date='2024-01-01',
    end_date='2024-06-30',
    overwrite=True,
    merge=False,
    plot=False,
    quiet=False,
)


In [ ]:
# 4a. Merge
gdf_cwc_pipe = hydroswift.merge_only(
    mode='cwc',
    input_dir='data/pipeline_cwc',
    output_dir='data/pipeline_cwc_merged',
)
gdf_cwc_pipe.head()


In [ ]:
# 4b. Plot
hydroswift.plot_only(
    mode='cwc',
    input_dir='data/pipeline_cwc',
    output_dir='data/pipeline_cwc_plots',
    moving_average=True,
    window=10,
)


### 11c. Download by specifying basins etc.

Use `wris.download()` / `cwc.download()` directly when inputs are already known:


In [ ]:
# WRIS — one-liner with everything
gdf_direct = hydroswift.wris.download(
    basin=['Godavari', 'Krishna'],
    variable=['discharge', 'sediment'],
    start_date='2024-01-01',
    end_date='2024-12-31',
    output_dir='data/wris_direct',
    format='csv',
    overwrite=True,
    merge=True,
    plot=True,
    quiet=False,
)
gdf_direct


In [ ]:
# CWC — one-liner
gdf_cwc_direct = hydroswift.cwc.download(
    basin=['Krishna', 'Godavari'],
    start_date='2024-01-01',
    end_date='2024-06-30',
    output_dir='data/cwc_direct',
    format='csv',
    overwrite=True,
    merge=True,
    plot=True,
    quiet=False,
)
gdf_cwc_direct


## 12. API Quick Reference Table

| Task | Method | Key Parameters |
|---|---|---|
| List WRIS variables | `hydroswift.wris.variables()` | — |
| List WRIS basins | `hydroswift.wris.basins()` | `variable=` (optional) |
| Discover WRIS stations | `hydroswift.wris.stations()` | `basin=`, `variable=`, `delay=` |
| List CWC basins | `hydroswift.cwc.basins()` | `refresh=` |
| Discover CWC stations | `hydroswift.cwc.stations()` | `basin=`, `river=`, `state=`, `station=`, `refresh=` |
| Table-driven download | `hydroswift.fetch(table)` | `output_dir`, `start_date`, `end_date`, `merge`, `plot`, `format`, `overwrite`, `quiet`, `delay`, `refresh` |
| Explicit WRIS download | `hydroswift.wris.download()` | `basin=`, `variable=`, `station=`, + output options |
| Explicit CWC download | `hydroswift.cwc.download()` | `basin=`, `station=`, + output options |
| Merge to GeoPackage | `hydroswift.merge_only()` | `mode=`, `input_dir=`, `output_dir=`, `variable=` |
| Generate plots | `hydroswift.plot_only()` | `mode=`, `input_dir=`, `output_dir=`, `variable=`, `plot_svg=`, `moving_average=`, `window=` |
| Reconcile CWC metadata | `hydroswift.cwc.reconcile_metadata()` | `write=False` |
| Print API help | `hydroswift.help()` | — |
| Print CLI help | `hydroswift.cli_help()` | — |
| Citation info | `hydroswift.cite()` | — |
| Coffee easter egg | `hydroswift.coffee()` | — |


## 13. Misc.

In [1]:
import hydroswift

In [2]:
hydroswift.cite()

====================================================================

HYDROSWIFT ⚡

Fast, unified workflows for hydrological data

Carbform • carbform.github.io

Version: 1.0.0 — Arctic Amsterdam

...

██╔══██║  ╚██╔╝  ██║  ██║██╔══██╗██║   ██║╚════██║██║███╗██║██║██╔══╝     ██║   

██║  ██║   ██║   ██████╔╝██║  ██║╚██████╔╝███████║╚███╔███╔╝██║██║        ██║   

╚═╝  ╚═╝   ╚═╝   ╚═════╝ ╚═╝  ╚═╝ ╚═════╝ ╚══════╝ ╚══╝╚══╝ ╚═╝╚═╝        ╚═╝   

====================================================================


    If you use HydroSwift in your research, please cite:

    Sarat, C., Dash, D., & Kumar, A. (2026).
    HydroSwift: Automated Retrieval of Hydrological Station Data
    from India-WRIS and CWC Portals.
    Journal of Open Source Software.

    Repository:
    https://github.com/carbform/swift
    


In [3]:
hydroswift.coffee()


        ( (
        ) )
        ........
        |      |]
        \      /
        `----'
        TIME FOR A COFFEE BREAK
    
Many kinds of monkeys have a strong taste for tea, coffee and spirituous liqueurs. - Charles Darwin


### `cwc.reconcile_metadata()`

This refreshes the packaged CWC station metadata against the live CWC API.
Normally not needed in day-to-day downloads, but useful for debugging stale metadata.


In [5]:
# Preview reconciliation without writing to disk
reconciled = hydroswift.cwc.reconcile_metadata(write=False)
reconciled


,code,name,river,basin,state,district,division,lat,lon,rl_zero,warning_level,danger_level,hfl,hfl_date
0,001-CDBNG,Shimoga,Tunga,Krishna,Karnataka,Shivamogga,"Cauvery Division, Bangaluru",13.926944,75.585000,558.33,566.7,567.25,570.18,2019-08-10
1,001-CDJPR,Dhareri,Chambal,Ganga,Madhya Pradesh,UJJAIN,"Chambal Division(CD), Jaipur",23.147500,75.508333,465.00,478.5,479.50,482.62,2023-09-17
2,001-DDASL,Tilaiya Reservoir,Barakar,Ganga,Jharkhand,HAZARIBAG,"Damodar Division(DD), Asansol",24.320000,85.520000,0.00,NaN,NaN,372.28,1986-07-06
3,001-ERDBWN,Balimundali,Burhabalang,Subernarekha,Odisha,MAYURBHANJ,"Eastern Rivers Division(ERD), Bhubaneswar",21.733889,86.633333,35.00,NaN,NaN,43.10,2013-01-13
4,001-HGDDDN,Badrinath,Alaknanda,Ganga,Uttarakhand,Chamoli,"Himalayan Ganga Division,Haridwar",30.748333,79.495833,3107.00,3112.0,3113.00,3112.70,2007-08-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1636,YETIKOPPAKA,YETIKOPPAKA,varaha,Ganga,Andhra Pradesh,VISAKHAPATNAM,"RO, WQL-2, DOWLAISWARAM",17.488333,82.742778,0.00,NaN,NaN,NaN,None
1637,bv000f5,DUDHNAI,Dudhnai,Brahmaputra,Assam,GOALPARA,"Middle Brahmaputra Division (MBD), Guwahati",25.980556,90.790556,40.00,NaN,NaN,51.27,2014-09-22
1638,dwris-01,Upper Rimbi,Rimbi Khola,Brahmaputra,Sikkim,GYALSINGH,"Sikkim Investigation Division, Gangtok",27.330833,88.151944,1575.00,NaN,NaN,1582.80,2019-08-07
1639,nbo-2-006-1207,Silgi at Kotrai,Narmada,Narmada,Madhya Pradesh,DINDORI,"Narmada Division(ND), Bhopal",22.985278,80.596111,509.00,NaN,NaN,514.98,2023-08-03


In [6]:
# To overwrite the packaged metadata file (only do this if you know what you're doing!).
# Not recommended unless you have a specific reason to refresh the metadata from 
# the live CWC API and want to save it locally:

hydroswift.cwc.reconcile_metadata(write=True)


,code,name,river,basin,state,district,division,lat,lon,rl_zero,warning_level,danger_level,hfl,hfl_date
0,001-CDBNG,Shimoga,Tunga,Krishna,Karnataka,Shivamogga,"Cauvery Division, Bangaluru",13.926944,75.585000,558.33,566.7,567.25,570.18,2019-08-10
1,001-CDJPR,Dhareri,Chambal,Ganga,Madhya Pradesh,UJJAIN,"Chambal Division(CD), Jaipur",23.147500,75.508333,465.00,478.5,479.50,482.62,2023-09-17
2,001-DDASL,Tilaiya Reservoir,Barakar,Ganga,Jharkhand,HAZARIBAG,"Damodar Division(DD), Asansol",24.320000,85.520000,0.00,NaN,NaN,372.28,1986-07-06
3,001-ERDBWN,Balimundali,Burhabalang,Subernarekha,Odisha,MAYURBHANJ,"Eastern Rivers Division(ERD), Bhubaneswar",21.733889,86.633333,35.00,NaN,NaN,43.10,2013-01-13
4,001-HGDDDN,Badrinath,Alaknanda,Ganga,Uttarakhand,Chamoli,"Himalayan Ganga Division,Haridwar",30.748333,79.495833,3107.00,3112.0,3113.00,3112.70,2007-08-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1636,YETIKOPPAKA,YETIKOPPAKA,varaha,Ganga,Andhra Pradesh,VISAKHAPATNAM,"RO, WQL-2, DOWLAISWARAM",17.488333,82.742778,0.00,NaN,NaN,NaN,None
1637,bv000f5,DUDHNAI,Dudhnai,Brahmaputra,Assam,GOALPARA,"Middle Brahmaputra Division (MBD), Guwahati",25.980556,90.790556,40.00,NaN,NaN,51.27,2014-09-22
1638,dwris-01,Upper Rimbi,Rimbi Khola,Brahmaputra,Sikkim,GYALSINGH,"Sikkim Investigation Division, Gangtok",27.330833,88.151944,1575.00,NaN,NaN,1582.80,2019-08-07
1639,nbo-2-006-1207,Silgi at Kotrai,Narmada,Narmada,Madhya Pradesh,DINDORI,"Narmada Division(ND), Bhopal",22.985278,80.596111,509.00,NaN,NaN,514.98,2023-08-03
